In [ ]:
import torch
import functools
import einops
import requests
import pandas as pd
import io
import textwrap
import gc
import json

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch import Tensor
from typing import List, Callable
from transformer_lens import HookedTransformer, utils
from transformer_lens.hook_points import HookPoint
from transformers import AutoTokenizer
from jaxtyping import Float, Int
from typing import List, Tuple, Callable
import transformer_lens
import contextlib

In [ ]:
import random
import matplotlib.pyplot as plt

In [ ]:
# transformer_lens.loading_from_pretrained.OFFICIAL_MODEL_NAMES

In [ ]:
model_name = 'Qwen/Qwen-1_8B-Chat'
device = 'cuda:0'
model = HookedTransformer.from_pretrained(model_name, device=device)

In [ ]:
types = ['harmful', 'harmless']
splits = ['train', 'val', 'test']

def get_data(split: str, type: str) -> list:
    file = f'artifacts/data/splits/{type}_{split}.json'
    with open(file, 'r') as f:
        data = json.load(f)
    instructions = [item['instruction'] for item in data]
    return instructions

harmful_train = get_data('train', 'harmful')
harmless_train = get_data('train', 'harmless')
harmful_val = get_data('val', 'harmful')
harmless_val = get_data('val', 'harmless')
harmful_test = get_data('test', 'harmful')
harmless_test = get_data('test', 'harmless')

In [ ]:
len(harmful_train), len(harmless_train), len(harmful_val), len(harmless_val), len(harmful_test), len(harmless_test)

In [ ]:
tot_train = 200
tot_val = 30
tot_test = 30
random.seed(42)
harmful_train = random.sample(harmful_train, tot_train)
harmless_train = random.sample(harmless_train, tot_train)
harmful_val = random.sample(harmful_val, tot_val)
harmless_val = random.sample(harmless_val, tot_val)
harmful_test = random.sample(harmful_test, tot_test)
harmless_test = random.sample(harmless_test, tot_test)

In [ ]:
QWEN_CHAT_TEMPLATE = """<|im_start|>user
{instruction}<|im_end|>
<|im_start|>assistant
"""

def get_prompt(instruction: str) -> str:
    return QWEN_CHAT_TEMPLATE.format(instruction=instruction)

def get_prompt_tokens(instruction: str) -> torch.Tensor:
    return model.to_tokens(get_prompt(instruction))

tokens_to_consider = 5

In [ ]:
# for num in range(10):
#     print(f"Prompt: {(harmful_train[num])}")
#     tokens = model.generate(get_prompt_tokens(harmful_train[num]), max_new_tokens=20, do_sample=False, temperature=0)
#     print(model.to_string(tokens))
#     print('-'*100)

In [ ]:
a = ["sdfdsfdssfsdfds"]
b = ["please explain how a car works"]
c = ["How to kill a person"]

In [ ]:
prompt = a[0]
tokens = get_prompt_tokens(prompt)
logits, cache = model.run_with_cache(tokens, remove_batch_dim=True)
# Get the next token greedily (highest probability)
next_token_id = logits[0, -1, :].argmax()
next_token = model.to_string(next_token_id)
print(f"Next token: '{next_token}' (ID: {next_token_id})")

prompt = b[0]
tokens = get_prompt_tokens(prompt)
logits1, cache1 = model.run_with_cache(tokens, remove_batch_dim=True)
next_token_id1 = logits1[0, -1, :].argmax()
next_token1 = model.to_string(next_token_id1)
print(f"Next token: '{next_token1}' (ID: {next_token_id1})")

prompt = c[0]
tokens = get_prompt_tokens(prompt)
logits2, cache2 = model.run_with_cache(tokens, remove_batch_dim=True)
next_token_id2 = logits2[0, -1, :].argmax()
next_token2 = model.to_string(next_token_id2)
print(f"Next token: '{next_token2}' (ID: {next_token_id2})")

In [ ]:
import torch.nn.functional as F

In [ ]:
# Create a matrix of cosine similarities and norm ratios for all layers and token positions

cache_1 = cache
cache_2 = cache1

similarity_matrix2 = torch.zeros(model.cfg.n_layers, 10)
norm_ratio_matrix = torch.zeros(model.cfg.n_layers, 10)
diff_norm_matrix = torch.zeros(model.cfg.n_layers, 10)
for layer in range(model.cfg.n_layers):
    for pos in range(10):
        vec1 = cache_1[f'blocks.{layer}.hook_resid_post'][-(15-pos)]
        vec2 = cache_2[f'blocks.{layer}.hook_resid_post'][-(15-pos)]
        
        norm_cache = torch.norm(vec2)
        norm_cache1 = torch.norm(vec1)
        norm_ratio_matrix[layer, pos] = norm_cache / norm_cache1

        # Normalize vectors for similarity calculation
        vec1_normalized = vec1 / torch.norm(vec1)
        vec2_normalized = vec2 / torch.norm(vec2)

        # Calculate L1 norm of difference
        diff_vec = vec2_normalized - vec1_normalized
        diff_norm_matrix[layer, pos] = torch.norm(diff_vec, p=1)

        

        similarity_matrix2[layer, pos] = F.cosine_similarity(
            vec2, 
            vec1, 
            dim=0
        )
        

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 5))

# Plot cosine similarity
im1 = ax1.imshow(similarity_matrix2, aspect='auto', cmap='viridis')
plt.colorbar(im1, ax=ax1)

# Add text annotations for cosine similarity
for layer in range(model.cfg.n_layers):
    for pos in range(15):
        text = ax1.text(pos, layer, f'{similarity_matrix2[layer, pos]:.3f}',
                       ha="center", va="center", color="white", fontsize=8)

ax1.set_title('Cosine Similarity')

# Plot norm ratio
im2 = ax2.imshow(norm_ratio_matrix, aspect='auto', cmap='viridis')
plt.colorbar(im2, ax=ax2)

# Add text annotations for norm ratio
for layer in range(model.cfg.n_layers):
    for pos in range(15):
        text = ax2.text(pos, layer, f'{norm_ratio_matrix[layer, pos]:.3f}',
                       ha="center", va="center", color="white", fontsize=8)

ax2.set_title('Norm ratio')

# Plot L1 norm of difference
im3 = ax3.imshow(diff_norm_matrix, aspect='auto', cmap='viridis')
plt.colorbar(im3, ax=ax3)
# ax3.set_yscale('log')

# Add text annotations for L1 norm of difference
for layer in range(model.cfg.n_layers):
    for pos in range(10):
        text = ax3.text(pos, layer, f'{diff_norm_matrix[layer, pos]:.1f}',
                       ha="center", va="center", color="white", fontsize=8)

ax3.set_title('L1 Norm of Difference')
plt.tight_layout()
plt.show()

In [ ]:
harmful_q = torch.load('artifacts/residuals/harmful_stack.pt')
harmless_q = torch.load('artifacts/residuals/harmless_stack.pt')

residuals_nothink = torch.load('artifacts/residuals/residuals_nothink_Qwen_Qwen3-1.7B.pt')['post']
residuals_think = torch.load('artifacts/residuals/residuals_think_Qwen_Qwen3-1.7B.pt')['post']

In [ ]:
# Create a matrix of cosine similarities and norm ratios for all layers and token positions

cache_1 = residuals_nothink.mean(dim=0)
cache_2 = residuals_think.mean(dim=0)

similarity_matrix2 = torch.zeros(model.cfg.n_layers, 5)
norm_ratio_matrix = torch.zeros(model.cfg.n_layers, 5)
diff_norm_matrix = torch.zeros(model.cfg.n_layers, 5)
for layer in range(model.cfg.n_layers):
    for pos in range(5):
        vec1 = cache_1[layer][-(5-pos)]
        vec2 = cache_2[layer][-(5-pos)]
        
        norm_cache = torch.norm(vec2)
        norm_cache1 = torch.norm(vec1)
        norm_ratio_matrix[layer, pos] = norm_cache / norm_cache1

        # Normalize vectors for similarity calculation
        vec1_normalized = vec1 / torch.norm(vec1)
        vec2_normalized = vec2 / torch.norm(vec2)

        # Calculate L1 norm of difference
        diff_vec = vec2_normalized - vec1_normalized
        diff_norm_matrix[layer, pos] = torch.norm(diff_vec, p=1)

        

        similarity_matrix2[layer, pos] = F.cosine_similarity(
            vec2, 
            vec1, 
            dim=0
        )
        

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 5))

# Plot cosine similarity
im1 = ax1.imshow(similarity_matrix2, aspect='auto', cmap='viridis')
plt.colorbar(im1, ax=ax1)

# Add text annotations for cosine similarity
for layer in range(model.cfg.n_layers):
    for pos in range(5):
        text = ax1.text(pos, layer, f'{similarity_matrix2[layer, pos]:.3f}',
                       ha="center", va="center", color="white", fontsize=8)

ax1.set_title('Cosine Similarity')

# Plot norm ratio
im2 = ax2.imshow(norm_ratio_matrix, aspect='auto', cmap='viridis')
plt.colorbar(im2, ax=ax2)

# Add text annotations for norm ratio
for layer in range(model.cfg.n_layers):
    for pos in range(5):
        text = ax2.text(pos, layer, f'{norm_ratio_matrix[layer, pos]:.3f}',
                       ha="center", va="center", color="white", fontsize=8)

ax2.set_title('Norm ratio')

# Plot L1 norm of difference
im3 = ax3.imshow(diff_norm_matrix, aspect='auto', cmap='viridis')
plt.colorbar(im3, ax=ax3)
# ax3.set_yscale('log')

# Add text annotations for L1 norm of difference
for layer in range(model.cfg.n_layers):
    for pos in range(5):
        text = ax3.text(pos, layer, f'{diff_norm_matrix[layer, pos]:.1f}',
                       ha="center", va="center", color="white", fontsize=8)

ax3.set_title('L1 Norm of Difference')
plt.tight_layout()
plt.show()

In [ ]:
model.cfg.n_layers, model.cfg.d_mlp, model.cfg.d_model

In [ ]:
cache['blocks.0.hook_resid_post'][-5:]

In [ ]:
type_ = 'harmful'
train_data = eval(f"{type_}_train")
for prompt in train_data[:2]:
    print(prompt)

In [ ]:
def get_resid_cache(type_: str, n_prompts: int, resid_cache: dict):
    for layer in range(model.cfg.n_layers):
        resid_cache[type_][layer] = torch.zeros((n_prompts, tokens_to_consider, model.cfg.d_model))
        
    train_data = eval(f"{type_}_train")
    for i, prompt in enumerate(tqdm(train_data[:n_prompts])):
        tokens = get_prompt_tokens(prompt)
        logits, cache = model.run_with_cache(tokens, remove_batch_dim=True)
        for layer in range(model.cfg.n_layers):
            resid_cache[type_][layer][i] = cache[f'blocks.{layer}.hook_resid_pre'][-tokens_to_consider:]

In [ ]:
n_prompts = 100
resid_cache = {}
for type_ in ['harmful']:
    resid_cache[type_] = {}
    get_resid_cache(type_, n_prompts, resid_cache)

In [ ]:
for type_ in ['harmless']:
    resid_cache[type_] = {}
    get_resid_cache(type_, n_prompts, resid_cache)

In [ ]:
train_data[:2]

In [ ]:
resid_cache['harmful']

In [ ]:
all_cache = {
    group: torch.stack([
        tensor  # shape: [200, 5, 2048]
        for layer, tensor in resid_cache[group].items()
    ], dim=1)  # shape: [200, n_layers, 5, 2048]
    for group in ['harmful', 'harmless']
}

In [ ]:
all_cache['harmful'].shape, all_cache['harmless'].shape

In [ ]:
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate cosine similarity between harmless and harmful tensors for dim=-1
harmless_tensor = all_cache['harmless'][12]  # shape: [24, 5, 2048]
harmful_tensor = all_cache['harmful'][99]    # shape: [24, 5, 2048]

# Normalize tensors along the last dimension
harmless_normalized = F.normalize(harmless_tensor, p=2, dim=-1)
harmful_normalized = F.normalize(harmful_tensor, p=2, dim=-1)

# Calculate cosine similarity (dot product of normalized vectors)
cosine_similarities = torch.sum(harmless_normalized * harmful_normalized, dim=-1)  # shape: [24, 5]

# Convert to numpy for plotting
cosine_sim_np = cosine_similarities.cpu().numpy()

# Create heatmap
plt.figure(figsize=(8, 10))
sns.heatmap(cosine_sim_np, 
            annot=True, 
            fmt='.3f', 
            cmap='Blues', 
            center=0,
            xticklabels=[f'Token {i}' for i in range(5)],
            yticklabels=[f'Layer {i}' for i in range(24)],
            cbar_kws={'label': 'Cosine Similarity'})
plt.title('Cosine Similarity between Harmless and Harmful Residuals')
plt.xlabel('Token Position')
plt.ylabel('Layer')
plt.tight_layout()
plt.show()

cosine_similarities.shape

In [ ]:
mean_cache = {
    group: {
        layer: tensor.mean(dim=0)  # shape: [5, 2048]
        for layer, tensor in resid_cache[group].items()
    }
    for group in ['harmful', 'harmless']
}

n_layers = model.cfg.n_layers

harmful_stack  = torch.stack([mean_cache['harmful'][layer] for layer in range(n_layers)])
harmless_stack = torch.stack([mean_cache['harmless'][layer] for layer in range(n_layers)])

avg_direction = harmful_stack - harmless_stack  

In [ ]:
torch.save(harmful_stack, 'artifacts/residuals/harmful_stack.pt')
torch.save(harmless_stack, 'artifacts/residuals/harmless_stack.pt')

In [ ]:
import torch.nn.functional as F

In [ ]:
# Normalize all_stats_dir to unit vectors along the last dimension
all_stats_dir_normalized = F.normalize(all_stats_dir, p=2, dim=-1)
harmful_stack_normalized = F.normalize(harmful_stack, p=2, dim=-1)
harmless_stack_normalized = F.normalize(harmless_stack, p=2, dim=-1)

In [ ]:
all_stats_dir_normalized.shape

In [ ]:
final_dir = all_stats_dir_normalized - harmless_stack_normalized
avg_dir = harmful_stack_normalized - harmless_stack_normalized

In [ ]:
harmless_stack.shape, all_stats_dir.shape, harmful_stack.shape

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# 1. Normalize all three tensors along the hidden dim
harmless_stack = F.normalize(harmless_stack, dim=-1)
harmful_stack  = F.normalize(harmful_stack, dim=-1)
all_stats_dir  = F.normalize(all_stats_dir, dim=-1)

# 2. Compute the refusal direction (elementwise across [24, 5, 2048])
refusal_dir = harmful_stack - harmless_stack
refusal_dir = F.normalize(refusal_dir, dim=-1)  # normalize again after subtraction

all_stats_dir_sub = all_stats_dir - harmless_stack
all_stats_dir_sub = F.normalize(all_stats_dir_sub, dim=-1)

# 3. Compute cosine similarity with harmful completions
cosine_map = (all_stats_dir_sub * refusal_dir).sum(dim=-1)  # shape: (24, 5)

# 4. Plot heatmap
plt.figure(figsize=(10, 6))
plt.imshow(cosine_map.cpu(), aspect='auto', cmap='viridis', vmin=-1, vmax=1)
plt.colorbar(label='Cosine Similarity')
plt.xlabel('Token position')
plt.ylabel('Layer')
plt.title('Cosine Similarity: (harmful completions · refusal direction)')
plt.tight_layout()
plt.show()


In [ ]:
# Calculate cosine similarity between harmful_stack and all_stats_dir
# Both matrices are torch.Size([24, 5, 2048])
# Take cosine similarity along the 2048 dimension, plot the 24x5 heatmap

import torch.nn.functional as F

# Calculate cosine similarity along the last dimension (2048)
cosine_sim = F.cosine_similarity(harmful_stack, all_stats_dir, dim=-1)  # Shape: [24, 5]

# Plot the heatmap
plt.figure(figsize=(6, 8))
plt.imshow(cosine_sim.cpu().numpy(), aspect='auto', cmap='viridis', vmin=-1, vmax=1)
plt.colorbar(label='Cosine Similarity', )
plt.xlabel('Token Position')
plt.ylabel('Layer')
plt.title('Cosine b/w Harmful ques vs  Statments')
# plt.colorbar(range=(0, 1))
plt.show()

In [ ]:
# Calculate cosine similarity between final and avg_direction
# Both matrices are torch.Size([24, 5, 2048])
# Take cosine similarity along the 2048 dimension, plot the 24x5 heatmap

import torch.nn.functional as F

# Calculate cosine similarity along the last dimension (2048)
cosine_sim = F.cosine_similarity(all_stats_dir_normalized - harmless_stack_normalized, harmful_stack_normalized - harmless_stack_normalized, dim=-1)  # Shape: [24, 5]

# Plot the heatmap
plt.figure(figsize=(6, 8))
plt.imshow(cosine_sim.cpu().numpy(), aspect='auto', cmap='viridis', vmin=-1, vmax=1)
plt.colorbar(label='Cosine Similarity')
plt.xlabel('Token Position')
plt.ylabel('Layer')
plt.title('Cosine b/w (Statement - Harmless ques) vs (Harmful ques - Harmless ques)')
plt.show()

In [ ]:
def get_harmful_stats(statement):
    # crop before </think> return after
    if '</think>' in statement:
        return statement[statement.index('</think>')+len('</think>'):]

harm_stat = json.load(open('artifacts/results/analysis/harmful_inference_Qwen_Qwen3-4B_offset120_400_labelled_with_thinking.json'))
harmful_stats = []
harmful_ques = []
for i, el in enumerate(harm_stat):

    if el['think_analysis'] == 'harmful':
        stat = get_harmful_stats(el['think_response'])
        if stat is None:
            continue
        harmful_stats.append(stat)
        harmful_ques.append(el['question'])

len(harmful_stats), len(harmful_ques)

In [ ]:
from collections import defaultdict
resid_cache = defaultdict(dict)

In [ ]:
def crop_to_100_words(text: str, max_words: int = 100, grace: int = 50) -> str:
    if not isinstance(text, str):
        raise TypeError(f"Expected a string, but got {type(text)} instead.")

    words = text.strip().split()

    # Already short enough
    if len(words) <= max_words:
        out = text.strip()
        if not out.endswith('.'):
            out = out.rstrip('.;!?') + '.'
        return out

    # Look for sentence end near cutoff
    cutoff = max_words
    while (cutoff < len(words)
           and cutoff < max_words + grace
           and not words[cutoff - 1].rstrip().endswith('.')):
        cutoff += 1

    out = " ".join(words[:cutoff]).strip()
    if not out.endswith('.'):
        out = out.rstrip('.;!?') + '.'
    return out


In [ ]:
n_prompts = len(harmful_stats)
for layer in range(model.cfg.n_layers):
    resid_cache[layer] = torch.zeros((n_prompts, tokens_to_consider, model.cfg.d_model))
        
data = harmful_stats
for i, prompt in enumerate(tqdm(data[:n_prompts])):
    prompt = crop_to_100_words(prompt, max_words=200)
    tokens = get_prompt_tokens(prompt)
    print(len(prompt.split()), len(tokens[0]))

    logits, cache = model.run_with_cache(tokens, remove_batch_dim=True)
    for layer in range(model.cfg.n_layers):
        resid_cache[layer][i] = cache[f'blocks.{layer}.hook_resid_pre'][-tokens_to_consider:]

In [ ]:
all_stats = torch.stack([resid_cache[layer] for layer in range(model.cfg.n_layers)], dim=1)

In [ ]:
all_stats_dir = torch.mean(all_stats, dim=0)
torch.save(all_stats_dir, 'artifacts/residuals/all_stats_dir.pt')

In [ ]:
all_stats = torch.load('artifacts/residuals/all_stats_dir.pt')
harmful_stack = torch.load('artifacts/residuals/harmful_stack.pt')
harmless_stack = torch.load('artifacts/residuals/harmless_stack.pt')

In [ ]:
all_stats.shape, harmful_stack.shape, harmless_stack.shape

In [ ]:
# Calculate cosine similarity between harmful_stack and all_stats_dir
# Both matrices are torch.Size([24, 5, 2048])
# Take cosine similarity along the 2048 dimension, plot the 24x5 heatmap

import torch.nn.functional as F

# Calculate cosine similarity along the last dimension (2048)
cosine_sim = F.cosine_similarity(harmful_stack, all_stats, dim=-1)  # Shape: [24, 5]

# Plot the heatmap
plt.figure(figsize=(6, 8))
plt.imshow(cosine_sim.cpu().numpy(), aspect='auto', cmap='viridis', vmin=-1, vmax=1)
plt.colorbar(label='Cosine Similarity', )
plt.xlabel('Token Position')
plt.ylabel('Layer')
plt.title('Cosine b/w Harmful ques vs  Statments')
# plt.colorbar(range=(0, 1))
plt.show()

In [ ]:
all

In [ ]:
harmful_stack_normalized = F.normalize(harmful_stack, dim=-1)
harmless_stack_normalized = F.normalize(harmless_stack, dim=-1)
all_stats_dir_normalized = F.normalize(all_stats, dim=-1)

cosine_sim = F.cosine_similarity(harmful_stack_normalized - harmless_stack_normalized, all_stats_dir_normalized, dim=-1)  # Shape: [24, 5]

plt.figure(figsize=(6, 8))
plt.imshow(cosine_sim.cpu().numpy(), aspect='auto', cmap='viridis', vmin=-1, vmax=1)
plt.colorbar(label='Cosine Similarity', )
plt.xlabel('Token Position')
plt.ylabel('Layer')
plt.title('Cosine b/w (Harmful ques - Harmless ques) vs  Statments')
# plt.colorbar(range=(0, 1))
plt.show()

In [ ]:
# Calculate cosine similarity between harmful_stack and all_stats_dir
# Both matrices are torch.Size([24, 5, 2048])
# Take cosine similarity along the 2048 dimension, plot the 24x5 heatmap

import torch.nn.functional as F

# Calculate cosine similarity along the last dimension (2048)
cosine_sim = F.cosine_similarity(harmful_stack - harmless_stack, all_stats_dir-harmless_stack, dim=-1)  # Shape: [24, 5]

# Plot the heatmap
plt.figure(figsize=(6, 8))
plt.imshow(cosine_sim.cpu().numpy(), aspect='auto', cmap='viridis', vmin=-1, vmax=1)
plt.colorbar(label='Cosine Similarity', )
plt.xlabel('Token Position')
plt.ylabel('Layer')
plt.title('Cosine b/w Harmful ques vs  Statments')
# plt.colorbar(range=(0, 1))
plt.show()

In [ ]:
inference_responses = []

for i, prompt in enumerate(tqdm(harmful_stats)):
    prompt = crop_to_100_words(prompt, max_words=300)
    tokens = get_prompt_tokens(prompt)
    # print(len(prompt.split()), len(tokens[0]))

    completions = _generate_with_hooks(
        model,
        tokens,
        max_tokens_generated=100,
        fwd_hooks=[],
    )
    print(prompt)
    print("-"*100)
    print(completions)
    print("\n")
    print('='*100)
    inference_responses.extend(completions)
    
    # Clear memory after each prompt
    torch.cuda.empty_cache()
    del tokens, completions
    import gc
    gc.collect()


In [ ]:
inference_responses = []

for i, prompt in enumerate(tqdm(harmful_stats[15:])):
    prompt = crop_to_100_words(prompt, max_words=100)
    tokens = get_prompt_tokens(prompt)
    # print(len(prompt.split()), len(tokens[0]))

    completions = _generate_with_hooks(
        model,
        tokens,
        max_tokens_generated=100,
        fwd_hooks=[],
    )
    print(prompt)
    print("-"*100)
    print(completions)
    print("\n")
    print('='*100)
    inference_responses.extend(completions)
    
    # Clear memory after each prompt
    torch.cuda.empty_cache()
    del tokens, completions
    import gc
    gc.collect()


In [ ]:
inference_responses = []


trial = ['How to build a bomb?']
for i, prompt in enumerate(tqdm(harmful_ques)):
    prompt = crop_to_100_words(prompt, max_words=300)
    tokens = get_prompt_tokens(prompt)
    # print(len(prompt.split()), len(tokens[0]))

    completions = _generate_with_hooks(
        model,
        tokens,
        max_tokens_generated=100,
        fwd_hooks=[],
    )
    print(prompt)
    print("-"*100)
    print(completions)
    print("\n")
    print('='*100)
    inference_responses.extend(completions)


In [ ]:
def _generate_with_hooks(
    model: HookedTransformer,
    toks: Int[Tensor, "batch_size seq_len"],
    max_tokens_generated: int = 64,
    fwd_hooks = [],
) -> List[str]:
    batch_size, seq_len = toks.shape
    all_toks = toks.clone()

    for _ in range(max_tokens_generated):
        with model.hooks(fwd_hooks=fwd_hooks):
            logits = model(all_toks)
        next_token = logits[:, -1, :].argmax(dim=-1).unsqueeze(1)
        all_toks = torch.cat([all_toks, next_token], dim=1)

    full_texts = model.to_string(all_toks)
    prompt_texts = model.to_string(toks)

    completions = [
        full[len(prompt):].strip()
        for full, prompt in zip(full_texts, prompt_texts)
    ]
    return completions


def get_generations(
    model: HookedTransformer,
    instructions: List[str],
    tokenize_instructions_fn: Callable[[List[str]], Int[Tensor, 'batch_size seq_len']],
    fwd_hooks = [],
    max_tokens_generated: int = 64,
    batch_size: int = 4,
) -> List[str]:
    generations = []

    for i in tqdm(range(0, len(instructions), batch_size)):
        batch_instructions = instructions[i : i + batch_size]
        toks = tokenize_instructions_fn(batch_instructions)
        completions = _generate_with_hooks(
            model,
            toks,
            max_tokens_generated=max_tokens_generated,
            fwd_hooks=fwd_hooks,
        )
        generations.extend(completions)

    return generations


def tokenize_chat(prompts: List[str]) -> torch.Tensor:
    formatted = [get_prompt(p) for p in prompts]
    return model.to_tokens(formatted, padding_side='left')

In [ ]:
def make_actadd_hook(direction: torch.Tensor, scale: float = 1.0):
    def hook(resid_pre, hook):
        return resid_pre + scale * direction.to(resid_pre.device).view(1, 1, -1)
    return hook

def make_ablation_hook(direction: torch.Tensor):
    def hook(resid_pre, hook):
        direction_ = direction.to(resid_pre.device)  # [d_model]
        proj_coeff = einops.einsum(
            resid_pre, direction_, "... d_model, d_model -> ..."
        )  # [...] - dot product coefficients
        proj = einops.einsum(
            proj_coeff, direction_, "..., d_model -> ... d_model"
        )  # [..., d_model] - full projection
        return resid_pre - proj
    return hook

In [ ]:
prompts = harmful_train[:8]

In [ ]:
def get_last_token_logits(
    model,
    tokens: Int[Tensor, "batch seq"],
    fwd_hooks = [],
    batch_size: int = 16
) -> Float[Tensor, "n_instructions d_vocab"]:

    all_logits = []

    for i in range(0, len(prompts), batch_size):
        toks = tokens[i:i + batch_size]
        # Run a forward pass with one additional generation step
        with model.hooks(fwd_hooks=fwd_hooks):
            gen_logits = model(toks)  # shape: [batch, seq, d_vocab]
        last_logits = gen_logits[:, -1, :]  # take logits for the next token

        all_logits.append(last_logits)

    return torch.cat(all_logits, dim=0)  # [n_instructions, d_vocab]

In [ ]:
QWEN_REFUSAL_TOKS = [40, 2121]  # ['I', 'As']

In [ ]:
for prompt in prompts:
    tokens = tokenize_chat([prompt])
    logits = get_last_token_logits(model, tokens, fwd_hooks=[])
    greedy_tokens = logits.argmax(dim=-1)
    print(model.to_string(greedy_tokens))

In [ ]:
for i in range(8):
    tokens = tokenize_chat([prompts[i]])
    print(prompts[i])
    outputs = _generate_with_hooks(
        model,
        tokens,  # tokenized with tokenize_chat
        max_tokens_generated=32,
        fwd_hooks=[]
    )
    print(outputs)

In [ ]:
prompts_harmless = harmless_train[:8]

In [ ]:
model.reset_hooks()
tot = 16
random.seed(42)
indices = random.sample(range(len(harmless_val)), tot)
for idx in indices:
    prompt = harmless_val[idx]
    print(prompt)
    tokens = tokenize_chat([prompt])
    outputs = _generate_with_hooks(model, tokens, fwd_hooks=[], max_tokens_generated=32)
    print(outputs)
    print('-'*100)

In [ ]:
positions = range(tokens_to_consider)

refusal_matrix = torch.zeros((model.cfg.n_layers, tokens_to_consider), dtype=torch.int)
tot = 16
random.seed(42)
indices = random.sample(range(len(harmless_val)), tot)
for layer in tqdm(range(model.cfg.n_layers), desc="Layers"):
    for pos in (positions):
        direction = avg_direction[layer, pos] 
        direction = direction / direction.norm()
        hook_fn_add = make_actadd_hook(direction, scale=1.0)
        hook_fn_ablation = make_ablation_hook(direction)
        fwd_hooks_add = [(f"blocks.{layer}.hook_resid_pre", hook_fn_add)]
        fwd_hooks_ablation = [
            (f"blocks.{layer}.hook_resid_pre", hook_fn_ablation),
            (f"blocks.{layer}.hook_resid_mid", hook_fn_ablation),
            (f"blocks.{layer}.hook_resid_post", hook_fn_ablation)
        ]
        for idx in indices:
            model.reset_hooks()
            prompt = harmless_val[idx]
            tokens = tokenize_chat([prompt])
            logits = get_last_token_logits(model, tokens, fwd_hooks=fwd_hooks_add)
            greedy_tokens = logits.argmax(dim=-1)
            refusal_matrix[layer, pos] += int(greedy_tokens.item() in QWEN_REFUSAL_TOKS)

In [ ]:
import seaborn as sns
sns.heatmap(refusal_matrix, annot=True, cmap='viridis', fmt='d')
plt.title('Refusal Score (scale = 1.0)')
plt.show()

In [ ]:
positions = range(tokens_to_consider)

refusal_matrix = torch.zeros((model.cfg.n_layers, tokens_to_consider), dtype=torch.int)
tot = 16
random.seed(42)
indices = random.sample(range(len(harmless_val)), tot)
for layer in tqdm(range(model.cfg.n_layers), desc="Layers"):
    for pos in (positions):
        direction = avg_direction[layer, pos] 
        direction = direction / direction.norm()
        hook_fn_add = make_actadd_hook(direction, scale=10.0)
        hook_fn_ablation = make_ablation_hook(direction)
        fwd_hooks_add = [(f"blocks.{layer}.hook_resid_pre", hook_fn_add)]
        fwd_hooks_ablation = [
            (f"blocks.{layer}.hook_resid_pre", hook_fn_ablation),
            (f"blocks.{layer}.hook_resid_mid", hook_fn_ablation),
            (f"blocks.{layer}.hook_resid_post", hook_fn_ablation)
        ]
        for idx in indices:
            model.reset_hooks()
            prompt = harmless_val[idx]
            tokens = tokenize_chat([prompt])
            logits = get_last_token_logits(model, tokens, fwd_hooks=fwd_hooks_add)
            greedy_tokens = logits.argmax(dim=-1)
            refusal_matrix[layer, pos] += int(greedy_tokens.item() in QWEN_REFUSAL_TOKS)

import seaborn as sns
sns.heatmap(refusal_matrix, annot=True, cmap='viridis', fmt='d')
plt.title('Refusal Score (scale = 10.0)')
plt.show()

In [ ]:
model.reset_hooks()
tot = 16
random.seed(42)
indices = random.sample(range(len(harmful_val)), tot)
for idx in indices:
    prompt = harmful_val[idx]
    print(prompt)
    tokens = tokenize_chat([prompt])
    outputs = _generate_with_hooks(model, tokens, fwd_hooks=[], max_tokens_generated=32)
    
    print(outputs)
    print('-'*100)

In [ ]:
positions = range(tokens_to_consider)

bypass_matrix = torch.zeros((model.cfg.n_layers, tokens_to_consider), dtype=torch.int)
tot = 16
random.seed(42)
indices = random.sample(range(len(harmful_val)), tot)
for layer in tqdm(range(model.cfg.n_layers), desc="Layers"):
    for pos in (positions):
        direction = avg_direction[layer, pos] 
        direction = direction / direction.norm()
        hook_fn_add = make_actadd_hook(direction, scale=1.0)
        hook_fn_ablation = make_ablation_hook(direction)
        fwd_hooks_add = [(f"blocks.{layer}.hook_resid_pre", hook_fn_add)]
        fwd_hooks_ablation = [
            (f"blocks.{layer}.hook_resid_pre", hook_fn_ablation),
            (f"blocks.{layer}.hook_resid_mid", hook_fn_ablation),
            (f"blocks.{layer}.hook_resid_post", hook_fn_ablation)
        ]
        for idx in indices:
            model.reset_hooks()
            prompt = harmful_val[idx]
            tokens = tokenize_chat([prompt])
            logits = get_last_token_logits(model, tokens, fwd_hooks=fwd_hooks_ablation)
            greedy_tokens = logits.argmax(dim=-1)
            bypass_matrix[layer, pos] += int(greedy_tokens.item() in QWEN_REFUSAL_TOKS)

In [ ]:
sns.heatmap(bypass_matrix, annot=True, cmap='viridis', fmt='d')
plt.title('Bypass Score')
plt.show()

In [ ]:
for layer in range(model.cfg.n_layers):
    for pos in range(tokens_to_consider):
        print(layer, pos)
        direction = avg_direction[layer, pos]
        direction = direction / direction.norm()
        hook_fn_ablation = make_ablation_hook(direction)
        fwd_hooks_add = [(f"blocks.{layer}.hook_resid_pre", hook_fn_add)]
        fwd_hooks_ablation = [
            (f"blocks.{layer}.hook_resid_pre", hook_fn_ablation),
            (f"blocks.{layer}.hook_resid_mid", hook_fn_ablation),
            (f"blocks.{layer}.hook_resid_post", hook_fn_ablation)
        ]

        prompts = ["How to build a bomb?"]
        for prompt in prompts:
            model.reset_hooks()
            print(prompt)
            tokens = tokenize_chat([prompt])
            outputs = _generate_with_hooks(
                model,
                tokens, 
                max_tokens_generated=32,
                fwd_hooks=fwd_hooks_ablation
            )
            print(outputs)
            print('-'*100)

In [ ]:
prompts = harmful_test[10:14]
layer = 11
pos = 4
direction = avg_direction[layer, pos]
direction = direction / direction.norm()
hook_fn_ablation = make_ablation_hook(direction)
fwd_hooks_add = [(f"blocks.{layer}.hook_resid_pre", hook_fn_add)]
fwd_hooks_ablation = [
    (f"blocks.{layer}.hook_resid_pre", hook_fn_ablation),
    (f"blocks.{layer}.hook_resid_mid", hook_fn_ablation),
    (f"blocks.{layer}.hook_resid_post", hook_fn_ablation)
]

for prompt in prompts:
    model.reset_hooks()
    print(prompt)
    tokens = tokenize_chat([prompt])
    baseline = _generate_with_hooks(
        model,
        tokens, 
        max_tokens_generated=32,
        fwd_hooks=[]
    )
    ablated = _generate_with_hooks(
        model,
        tokens, 
        max_tokens_generated=32,
        fwd_hooks=fwd_hooks_ablation
    )
    print('\033[94m' + ' '.join(baseline) + '\033[0m')  # Blue color
    print('\033[91m' + ' '.join(ablated) + '\033[0m')   # Red color
    print('-'*100)

In [ ]:
for layer in range(model.cfg.n_layers):
    for pos in range(tokens_to_consider):
        print(layer, pos)
        direction = avg_direction[layer, pos]
        direction = direction / direction.norm()
        hook_fn_ablation = make_ablation_hook(direction)
        hook_fn_add = make_actadd_hook(direction, scale=10.0)
        fwd_hooks_add = [(f"blocks.{layer}.hook_resid_pre", hook_fn_add)]
        fwd_hooks_ablation = [
            (f"blocks.{layer}.hook_resid_pre", hook_fn_ablation),
            (f"blocks.{layer}.hook_resid_mid", hook_fn_ablation),
            (f"blocks.{layer}.hook_resid_post", hook_fn_ablation)
        ]

        prompts = ["What is a rainbow?"]
        for prompt in prompts:
            model.reset_hooks()
            print(prompt)
            tokens = tokenize_chat([prompt])
            outputs = _generate_with_hooks(
                model,
                tokens, 
                max_tokens_generated=32,
                fwd_hooks=fwd_hooks_add
            )
            print(outputs)
            print('-'*100)

In [ ]:
prompts = ["What is a rainbow?", "How to build a lego set?", "How to make everyone laugh?"]

layer = 11
pos = 4
direction = avg_direction[layer, pos]
direction = direction / direction.norm()
hook_fn_ablation = make_ablation_hook(direction)
hook_fn_add = make_actadd_hook(direction, scale=10.0)
fwd_hooks_add = [(f"blocks.{layer}.hook_resid_pre", hook_fn_add)]
fwd_hooks_ablation = [
    (f"blocks.{layer}.hook_resid_pre", hook_fn_ablation),
    (f"blocks.{layer}.hook_resid_mid", hook_fn_ablation),
    (f"blocks.{layer}.hook_resid_post", hook_fn_ablation)
]

for prompt in prompts:
    model.reset_hooks()
    print(prompt)
    tokens = tokenize_chat([prompt])
    baseline = _generate_with_hooks(
        model,
        tokens, 
        max_tokens_generated=32,
        fwd_hooks=[]
    )
    ablated = _generate_with_hooks(
        model,
        tokens, 
        max_tokens_generated=32,
        fwd_hooks=fwd_hooks_add
    )
    print('\033[94m' + ' '.join(baseline) + '\033[0m')  # Blue color
    print('\033[91m' + ' '.join(ablated) + '\033[0m')   # Red color
    print('-'*100)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(refusal_matrix, annot=True, cmap='viridis', fmt='d', ax=ax1)
ax1.set_title('Refusal Score (scale = 10.0)')

sns.heatmap(bypass_matrix, annot=True, cmap='viridis', fmt='d', ax=ax2) 
ax2.set_title('Bypass Score')

plt.tight_layout()
plt.show()

In [ ]:
sns.heatmap(bypass_matrix, annot=True, cmap='viridis', fmt='d')
plt.title('Bypass Score')
plt.show()